In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_selection import RFECV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [2]:
FEAT_PATH       = "../data/features.parquet"
MASKS_PATH      = "../data/feature_masks.pkl"
FORECAST_POINTS = 8
SOURCE_COL      = "sum_1h"   # колонка, из которой делаются сдвиги → target_step_1..8
SAMPLE_SIZE     = 300_000
RANDOM_STATE    = 42
RIDGE_ALPHA     = 4.0
CV_FOLDS        = 3

In [3]:
# --- Загрузка данных ---
df = pd.read_parquet(FEAT_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
print("Загружено:", df.shape)
print("Колонки:", df.columns.tolist())

Загружено: (1702000, 218)
Колонки: ['route_id', 'timestamp', 'status_1', 'status_2', 'status_3', 'status_4', 'status_5', 'status_6', 'sum_1h', 'status_1-3_sum', 'status_4-6_sum', 'status_1-3_all_prop', 'status_4-6_all_prop', 'status_1_1-3_prop', 'status_2_1-3_prop', 'status_3_1-3_prop', 'status_4_4-6_prop', 'status_5_4-6_prop', 'status_6_4-6_prop', 'mean_sum_1h_lag_327', 'mean_sum_1h_lag_328', 'mean_sum_1h_lag_329', 'mean_sum_1h_lag_330', 'mean_sum_1h_lag_331', 'mean_sum_1h_lag_332', 'mean_sum_1h_lag_333', 'mean_sum_1h_lag_334', 'mean_sum_1h_lag_335', 'mean_sum_1h_lag_336', 'status_1_lag_1', 'status_2_lag_1', 'status_3_lag_1', 'status_4_lag_1', 'status_5_lag_1', 'status_6_lag_1', 'sum_1h_lag_1', 'status_1_lag_2', 'status_2_lag_2', 'status_3_lag_2', 'status_4_lag_2', 'status_5_lag_2', 'status_6_lag_2', 'sum_1h_lag_2', 'status_1_lag_3', 'status_2_lag_3', 'status_3_lag_3', 'status_4_lag_3', 'status_5_lag_3', 'status_6_lag_3', 'sum_1h_lag_3', 'status_1_lag_4', 'status_2_lag_4', 'status_3_l

In [4]:
# --- Создание таргетов (сдвиг SOURCE_COL на 1..8 шагов вперёд внутри маршрута) ---
route_group = df.groupby("route_id", sort=False)
FUTURE_TARGET_COLS = [f"target_step_{s}" for s in range(1, FORECAST_POINTS + 1)]

for step in range(1, FORECAST_POINTS + 1):
    df[f"target_step_{step}"] = route_group[SOURCE_COL].shift(-step)

df = df.dropna()
print("После dropna:", df.shape)

После dropna: (1502000, 226)


In [5]:
# --- Определение признаков ---
# route_id и timestamp — мета, не признаки
# sum_1h (lag 0) включаем: это текущее наблюдение, доступно на тесте, не утечка
# route_mean уже есть в features.parquet и покрывает роль LOO-среднего таргета
META_COLS = ["route_id", "timestamp"]

FEATURE_COLS = [
    c for c in df.columns
    if c not in META_COLS + FUTURE_TARGET_COLS
]
print(f"Признаков: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

Признаков: 216
['status_1', 'status_2', 'status_3', 'status_4', 'status_5', 'status_6', 'sum_1h', 'status_1-3_sum', 'status_4-6_sum', 'status_1-3_all_prop', 'status_4-6_all_prop', 'status_1_1-3_prop', 'status_2_1-3_prop', 'status_3_1-3_prop', 'status_4_4-6_prop', 'status_5_4-6_prop', 'status_6_4-6_prop', 'mean_sum_1h_lag_327', 'mean_sum_1h_lag_328', 'mean_sum_1h_lag_329', 'mean_sum_1h_lag_330', 'mean_sum_1h_lag_331', 'mean_sum_1h_lag_332', 'mean_sum_1h_lag_333', 'mean_sum_1h_lag_334', 'mean_sum_1h_lag_335', 'mean_sum_1h_lag_336', 'status_1_lag_1', 'status_2_lag_1', 'status_3_lag_1', 'status_4_lag_1', 'status_5_lag_1', 'status_6_lag_1', 'sum_1h_lag_1', 'status_1_lag_2', 'status_2_lag_2', 'status_3_lag_2', 'status_4_lag_2', 'status_5_lag_2', 'status_6_lag_2', 'sum_1h_lag_2', 'status_1_lag_3', 'status_2_lag_3', 'status_3_lag_3', 'status_4_lag_3', 'status_5_lag_3', 'status_6_lag_3', 'sum_1h_lag_3', 'status_1_lag_4', 'status_2_lag_4', 'status_3_lag_4', 'status_4_lag_4', 'status_5_lag_4', 's

In [6]:
# --- Сабсэмпл для скорости RFECV ---
df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).copy()
print(f"Размер сабсэмпла: {len(df_sample):,}")

Размер сабсэмпла: 300,000


In [7]:
# --- RFECV для каждого из 8 таргетов ---
# Таргет для step i: target_step_i = sum_1h сдвинутый на -i внутри маршрута
# Признаки: FEATURE_COLS (включает sum_1h lag 0 и route_mean, без LOO-средних)
# Маски сохраняются по ходу — можно прервать и воспользоваться промежуточными результатами

feature_masks = {}

for step in range(1, FORECAST_POINTS + 1):
    target_col = f"target_step_{step}"

    X = df_sample[FEATURE_COLS].values
    y = df_sample[target_col].values

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    estimator = Ridge(alpha=RIDGE_ALPHA)
    rfecv = RFECV(
        estimator=estimator,
        step=1,
        cv=CV_FOLDS,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        verbose=0,
    )
    rfecv.fit(X_scaled, y)

    mask     = rfecv.support_
    selected = [f for f, m in zip(FEATURE_COLS, mask) if m]

    feature_masks[step] = {
        "feature_names": FEATURE_COLS,
        "mask": mask,
        "selected": selected,
        "n_features_in": len(FEATURE_COLS),
        "n_selected": int(mask.sum()),
    }

    # Сохраняем после каждого шага — результаты не потеряются при прерывании
    joblib.dump(feature_masks, MASKS_PATH)
    print(f"step {step:2d}: отобрано {mask.sum():3d} / {len(FEATURE_COLS)} признаков  → сохранено")

print("\nГотово.")

step  1: отобрано 204 / 216 признаков  → сохранено
step  2: отобрано 212 / 216 признаков  → сохранено
step  3: отобрано 155 / 216 признаков  → сохранено
step  4: отобрано 131 / 216 признаков  → сохранено
step  5: отобрано 127 / 216 признаков  → сохранено
step  6: отобрано 121 / 216 признаков  → сохранено
step  7: отобрано 146 / 216 признаков  → сохранено
step  8: отобрано 168 / 216 признаков  → сохранено

Готово.


In [8]:
# --- Итоговая сводка ---
print(f"Маски сохранены → {MASKS_PATH}\n")
print("Сводка по отобранным признакам:")
for step, info in feature_masks.items():
    print(f"  step {step:2d}: {info['n_selected']:3d}/{info['n_features_in']} | {info['selected']}")

Маски сохранены → ../data/feature_masks.pkl

Сводка по отобранным признакам:
  step  1: 204/216 | ['status_1', 'status_2', 'status_3', 'status_4', 'status_5', 'status_6', 'sum_1h', 'status_1-3_sum', 'status_4-6_sum', 'status_1-3_all_prop', 'status_4-6_all_prop', 'status_1_1-3_prop', 'status_2_1-3_prop', 'status_3_1-3_prop', 'mean_sum_1h_lag_327', 'mean_sum_1h_lag_328', 'mean_sum_1h_lag_329', 'mean_sum_1h_lag_330', 'mean_sum_1h_lag_331', 'mean_sum_1h_lag_332', 'mean_sum_1h_lag_333', 'mean_sum_1h_lag_334', 'mean_sum_1h_lag_335', 'mean_sum_1h_lag_336', 'status_1_lag_1', 'status_2_lag_1', 'status_3_lag_1', 'status_4_lag_1', 'status_5_lag_1', 'status_6_lag_1', 'sum_1h_lag_1', 'status_1_lag_2', 'status_2_lag_2', 'status_3_lag_2', 'status_4_lag_2', 'status_5_lag_2', 'status_6_lag_2', 'sum_1h_lag_2', 'status_1_lag_3', 'status_2_lag_3', 'status_3_lag_3', 'status_4_lag_3', 'status_5_lag_3', 'status_6_lag_3', 'sum_1h_lag_3', 'status_1_lag_4', 'status_2_lag_4', 'status_3_lag_4', 'status_4_lag_4', 